## What Is Memory Routing?

### The core idea in one sentence
Analyse every incoming message for **content type** and **intent**, then dispatch it to the correct memory store(s) for reading and writing — rather than blindly querying all stores or writing to all stores on every turn.

---

### The mental model — an air traffic controller

An airport has multiple runways, terminals, and gates. When a plane arrives, the controller does not send it to all runways simultaneously. They look at the plane type, destination, size, and current traffic — then route it to the one correct runway and gate.

Memory routing is the same: each incoming message has a type and intent. The router looks at both and dispatches to exactly the right memory stores — not all of them.

```
"What is my current salary?"          --> ENTITY STORE (structured current fact)
"What did we decide last April?"       --> EPISODIC STORE (past session query)
"How does a SIP work?"                 --> VECTOR STORE (general knowledge retrieval)
"I just changed jobs to TCS"           --> ENTITY STORE (update) + VECTOR STORE (write)
"Never recommend equity to me again"   --> PROCEDURAL STORE (hard constraint write)
"I'm worried about market volatility"  --> SEMANTIC STORE (behavioural pattern read)
```

---

### Why routing matters — the cost of getting it wrong

**Without routing (query everything):**
- Entity store: 150 tokens injected
- Vector store: 200 tokens injected
- Episodic store: 250 tokens injected
- Semantic store: 200 tokens injected
- Reflection store: 150 tokens injected
- **Total: 950 tokens on every turn, regardless of relevance**

**With routing (query relevant stores only):**
- "How does a SIP work?" → vector store only: **200 tokens**
- "What is my salary?" → entity store only: **150 tokens**
- **60-80% token reduction. Same answer quality.**

---

### Two dimensions of routing

**READ routing** — which stores to query before generating a response:
- Fact questions → entity store
- Historical questions → episodic store
- General knowledge → vector store
- Behavioural questions → semantic store
- All investment advice → procedural + reflection store (always)

**WRITE routing** — which stores to update after receiving new information:
- User states a new fact → entity store update
- User states a preference or constraint → procedural store write
- Session ends → episodic store write + reflection generation
- Pattern confirmed across sessions → semantic store update

---

### Point 1 — Intent classification is the router's core mechanism

Every message is classified into an intent:
- `FACT_QUERY` — asking for a specific known fact
- `HISTORY_QUERY` — asking about a past session or event
- `KNOWLEDGE_QUERY` — asking about general financial concepts
- `FACT_UPDATE` — sharing new personal information
- `CONSTRAINT_UPDATE` — stating a hard rule or preference
- `ADVICE_REQUEST` — asking for a recommendation
- `EMOTIONAL_SIGNAL` — expressing anxiety, frustration, or uncertainty
- `GENERAL_CHAT` — small talk or non-financial query

Each intent maps to a READ set and a WRITE set. The router does not query beyond the mapped set.

---

### Point 2 — Routing can be rule-based or LLM-based

**Rule-based routing**: keyword and pattern matching. Fast, deterministic, zero cost. Works well for structured domains like financial advice where query types are predictable. Limitation: brittle — misses edge cases and ambiguous queries.

**LLM-based routing**: the router calls a cheap model (gpt-4o-mini) to classify intent and determine which stores to access. Slower by one call, but handles ambiguity and multi-intent messages correctly.

**Hybrid (our implementation)**: rule-based for high-confidence patterns (keyword matching), LLM-based fallback for ambiguous cases. Best of both worlds — speed where possible, accuracy where needed.

---

### Point 3 — Multi-intent messages need fan-out routing

Some messages have multiple intents simultaneously:

> *"I just changed jobs to TCS at ₹1,50,000 — given my conservative profile, what should I do with the salary increase?"*

This is: FACT_UPDATE (new salary) + ADVICE_REQUEST (what to do). The router must:
1. Write to entity store (update employer + salary)
2. Read from entity store (confirm current profile)
3. Read from procedural store (apply conservative rules)
4. Read from reflection store (check past mistakes)

Fan-out routing handles this by returning multiple intents and merging the store sets.

---

### Point 4 — Always-inject stores vs routed stores

Some stores should always be injected regardless of routing:
- **Procedural store**: operational rules should always shape every response
- **Reflection store (CRITICAL)**: compliance violations should always be visible

Some stores should only be injected when routing says so:
- **Episodic store**: only for history queries — wasteful for general advice
- **Vector store**: only for semantic knowledge queries — not for fact lookups

The router distinguishes between these two categories.

---

## Trade-offs

| | |
|---|---|
| ✅ 60-80% token reduction vs querying all stores | ❌ Misclassification sends query to wrong store |
| ✅ Faster responses — fewer retrieval calls | ❌ LLM-based routing adds one extra call |
| ✅ Reduces noise — only relevant context injected | ❌ Multi-intent detection adds complexity |
| ✅ Scales gracefully as more stores are added | ❌ Routing rules require domain expertise to write |

## Production Verdict

> **Essential once you have more than two memory stores. Not optional at scale.**
>
> Without routing, adding more memory stores makes the agent worse — more noise, more tokens, more confusion. With routing, each new store is additive — it improves the agent for the queries that need it without affecting the ones that do not.

---

What it is: Memory Routing classifies each memory operation by content type and routes it to the correct specialized store (episodic, semantic, or procedural).

When you need it: You run multiple memory backends and need an intelligent dispatcher instead of querying all stores on every request.

The trade-off: Every operation triggers a 200-500ms LLM classification call, and misrouting sends data to the wrong store silently.

Closest alternative in this repo: 13 Hierarchical Memory Layers routes by access frequency across tiers, not by content type across stores.

### Foundational information : 

Direct every memory read and write to the right specialized store using an intelligent dispatcher.

Think of a large office with separate filing cabinets. One holds client conversations. Another holds company policies. A third holds step-by-step procedures. When someone needs to file a document or retrieve one, they first decide which cabinet to open. That decision is routing.

As agent memory systems grow, they split into specialized stores. You might keep an episodic store for conversation histories, a semantic store for factual knowledge, and a procedural store for learned skills. Each store has its own data format and retrieval strategy. The challenge: how does the agent know which store to use for a given piece of information?

Without explicit routing, agents dump everything into one big store. They lose the benefits of specialization. Memory routing solves this by adding a dispatcher layer between the agent and its stores. For writes, the dispatcher classifies incoming content (is this a fact, a procedure, or an experience?) and sends it to the right store. For reads, it analyzes the query's intent and searches the relevant store or stores. When multiple stores match, it merges and re-ranks the results.

In this notebook you'll build:

Three specialized memory stores (episodic, semantic, procedural).

An LLM-powered router that classifies content and queries by type.

Write and read pipelines that direct operations to the correct store.

A fallback strategy for ambiguous queries that span multiple stores.


### Key Concepts

Memory type classification: Sorting incoming content into categories like episodic, semantic, or procedural. The classifier can be an LLM call, keyword rules, or a small trained model.

Episodic memory: Memories of events and experiences. "Yesterday's meeting," "the user's last request," or "the error that happened at 3 PM."

Semantic memory: Memories of facts and general knowledge. "The API rate limit is 1,000 requests per minute." "Python uses indentation for blocks."

Procedural memory: Memories of how to do things. Step-by-step instructions, workflows, and recipes. "To deploy: run tests, open a PR, merge to main."

Read routing: Analyzing a query's intent to pick which store(s) to search. "What happened yesterday?" routes to episodic. "What's our refund policy?" routes to semantic.

Write routing: Directing new content to the correct store with the right formatting. A conversation event gets a timestamp. A fact gets entity tags.

Store registry: A catalog of available memory stores. It maps each memory type to a store with its description and capabilities.

Fallback strategy: What happens when the router can't classify with confidence. Common options: search all stores and merge results, or default to a general-purpose store.

### How It Works

The router sends incoming content or query text to an LLM classifier (a model that picks the right category).

The classifier returns one or more memory types: episodic, semantic, or procedural.
For writes, the router creates a memory entry and stores it in each matching specialized store.

For reads, the router searches only the classified store(s) and scores results by keyword overlap.

Results from multiple stores are merged and deduplicated (removing repeated entries) before returning.

For ambiguous queries, a fallback mode searches all stores at once and merges the results.


In short ............

The Agent sends every memory operation (read or write) to the Memory Router. The router runs a classifier to determine the content type. For writes, it sends the content to the matching specialized store: Episodic (conversations and events), Semantic (facts and knowledge), or Procedural (skills and workflows). For reads, the router may query one or several stores based on the query's intent. Results pass through a Merge & Re-rank step that combines and scores items from different stores. The Store Registry tells the router what each store contains and how to use it.

In [1]:
!pip install openai tiktoken --quiet

In [2]:
import json, os, time, re
from datetime import datetime, timezone
from typing import List, Dict, Optional, Set, Tuple, Any
from dataclasses import dataclass, field
from enum import Enum

import openai
import tiktoken

In [3]:
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), "API key missing."

client        = openai.OpenAI(api_key=OPENAI_API_KEY)
CHAT_MODEL    = "gpt-4o"
ROUTER_MODEL  = "gpt-4o-mini"
# Routing is a classification task — mini is the right model.
# Fast, cheap, sufficient accuracy for intent detection.
TOKENISER     = tiktoken.get_encoding("o200k_base")

print(f"Chat model  : {CHAT_MODEL}")
print(f"Router model: {ROUTER_MODEL}")

Key loaded from environment variable.
Chat model  : gpt-4o
Router model: gpt-4o-mini


---
## Part 1 — Memory Store Registry and Intent Classification

In [4]:
class MemoryStore(str, Enum):
    """
    All memory stores available in the FinCoach system.
    Each is a distinct data store with different retrieval characteristics.
    """
    RECENT_BUFFER  = "recent_buffer"   # Short-term: current session turns (Techniques 01-05)
    VECTOR_STORE   = "vector_store"    # Message embeddings in ChromaDB (Technique 06)
    ENTITY_PROFILE = "entity_profile"  # Structured current-state facts (Technique 07)
    EPISODIC_STORE = "episodic_store"  # Session-level narrative records (Technique 09)
    SEMANTIC_STORE = "semantic_store"  # Distilled behavioural patterns (Technique 10)
    PROCEDURAL_STORE = "procedural_store"  # Learned operational rules (Technique 11)
    REFLECTION_STORE = "reflection_store"  # Self-critique notes (Technique 16)


class QueryIntent(str, Enum):
    """
    Classification of what the user is trying to do with their message.
    The router maps each intent to a set of stores to read from and write to.
    """
    FACT_QUERY       = "fact_query"       # "What is my salary?"
    HISTORY_QUERY    = "history_query"    # "What did we decide last April?"
    KNOWLEDGE_QUERY  = "knowledge_query"  # "How does a SIP work?"
    ADVICE_REQUEST   = "advice_request"   # "What should I do with my FD?"
    FACT_UPDATE      = "fact_update"      # "My new salary is X"
    CONSTRAINT_UPDATE = "constraint_update" # "Never recommend equity to me"
    EMOTIONAL_SIGNAL = "emotional_signal" # "I'm worried about markets"
    GENERAL_CHAT     = "general_chat"     # "Thanks!"


@dataclass
class RoutingDecision:
    """
    The output of the router for a single message.
    Specifies which stores to READ from and which to WRITE to.
    """
    message: str
    # The original user message.

    intents: List[str]
    # One or more QueryIntent values — a message can have multiple intents.

    read_stores: Set[str]
    # Which stores to query for context before generating the response.

    write_stores: Set[str]
    # Which stores to update after processing this message.

    routing_method: str
    # 'rule_based' or 'llm_based' — how this decision was made.

    confidence: float
    # Router's confidence in this classification (0.0 to 1.0).

    reasoning: str
    # Why the router made this decision — for debugging and audit.

    estimated_tokens_saved: int = 0
    # How many tokens were saved vs querying all stores.

    def token_cost_of_all_stores(self, store_token_estimates: Dict[str, int]) -> int:
        """What querying ALL stores would cost."""
        return sum(store_token_estimates.values())

    def token_cost_of_routed_stores(self, store_token_estimates: Dict[str, int]) -> int:
        """What querying only the ROUTED stores costs."""
        return sum(
            store_token_estimates.get(s, 0)
            for s in self.read_stores
        )

    def display(self) -> None:
        """Print the routing decision clearly."""
        print(f"  ROUTING DECISION ({self.routing_method}, conf={self.confidence:.0%})")
        print(f"  Intent(s)   : {', '.join(self.intents)}")
        print(f"  Read from   : {', '.join(sorted(self.read_stores)) or 'none'}")
        print(f"  Write to    : {', '.join(sorted(self.write_stores)) or 'none'}")
        print(f"  Reasoning   : {self.reasoning}")
        if self.estimated_tokens_saved > 0:
            print(f"  Tokens saved: ~{self.estimated_tokens_saved} vs querying all stores")


print("MemoryStore, QueryIntent, RoutingDecision defined.")

MemoryStore, QueryIntent, RoutingDecision defined.


---
## Part 2 — The Routing Table

The routing table maps each intent to its read and write store sets. This is the domain knowledge encoded into the router.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# ALWAYS-INJECT STORES
# These are included on EVERY API call regardless of routing.
# They contain operational rules and compliance notes that must always be visible.
# ─────────────────────────────────────────────────────────────────────────────
ALWAYS_READ = {
    MemoryStore.RECENT_BUFFER,
    # Always include the recent conversation — baseline for coherence.
    MemoryStore.PROCEDURAL_STORE,
    # Always include learned operational rules — they shape every response.
    MemoryStore.REFLECTION_STORE,
    # Always include active reflection notes — compliance and improvement.
}
# Note: ENTITY_PROFILE is NOT always-inject here — it is expensive (~150 tokens)
# and not always relevant (e.g. for knowledge queries about SIP mechanics).
# Instead it is injected for advice requests and fact queries.


# ─────────────────────────────────────────────────────────────────────────────
# INTENT TO STORE ROUTING TABLE
# Maps each intent to: {read_stores, write_stores}
# ─────────────────────────────────────────────────────────────────────────────
ROUTING_TABLE: Dict[str, Dict[str, Set[str]]] = {

    QueryIntent.FACT_QUERY: {
        # "What is my salary?" / "What is my risk profile?"
        # Needs: the structured profile — entity store.
        # Does NOT need: episodic store, vector store (irrelevant for direct lookups).
        "read":  {MemoryStore.ENTITY_PROFILE},
        "write": set(),  # Pure read — no writes needed.
    },

    QueryIntent.HISTORY_QUERY: {
        # "What did we decide last April?" / "Did I ever mention my FD before?"
        # Needs: episodic store for session-level recall,
        #        vector store for message-level fuzzy recall.
        # Does NOT need: entity profile (query is about the past, not current state).
        "read":  {MemoryStore.EPISODIC_STORE, MemoryStore.VECTOR_STORE},
        "write": set(),
    },

    QueryIntent.KNOWLEDGE_QUERY: {
        # "How does a SIP work?" / "What is the difference between PPF and NPS?"
        # Needs: vector store (similar past discussions on this topic).
        # Does NOT need: entity profile (generic question, not personal).
        # Does NOT need: episodic store (not a history question).
        "read":  {MemoryStore.VECTOR_STORE},
        "write": {MemoryStore.VECTOR_STORE},  # Store the Q&A for future retrieval.
    },

    QueryIntent.ADVICE_REQUEST: {
        # "What should I do with my FD maturity?" / "Should I start a SIP?"
        # Needs everything — this is the high-stakes case.
        # Entity profile: knows current facts.
        # Semantic store: knows user patterns and risk profile.
        # Episodic store: knows what was decided before.
        # Vector store: similar past discussions.
        "read":  {
            MemoryStore.ENTITY_PROFILE,
            MemoryStore.SEMANTIC_STORE,
            MemoryStore.EPISODIC_STORE,
            MemoryStore.VECTOR_STORE,
        },
        "write": {MemoryStore.VECTOR_STORE},  # Store the advice given.
    },

    QueryIntent.FACT_UPDATE: {
        # "My new salary is ₹1,50,000" / "I changed jobs to TCS"
        # Read: entity profile (get current state before update).
        # Write: entity profile (update the fact), vector store (store the message).
        "read":  {MemoryStore.ENTITY_PROFILE},
        "write": {MemoryStore.ENTITY_PROFILE, MemoryStore.VECTOR_STORE},
    },

    QueryIntent.CONSTRAINT_UPDATE: {
        # "Never recommend equity to me" / "I prefer action lists at the end"
        # Read: procedural store (check for conflicts with existing rules).
        # Write: procedural store (add/update the rule).
        "read":  {MemoryStore.PROCEDURAL_STORE},
        "write": {MemoryStore.PROCEDURAL_STORE, MemoryStore.ENTITY_PROFILE},
        # Also write to entity profile — constraints are structured facts too.
    },

    QueryIntent.EMOTIONAL_SIGNAL: {
        # "I'm really worried about markets" / "I feel anxious about investing"
        # Read: semantic store (know how user typically handles anxiety).
        #       entity profile (personalise the reassurance).
        # Write: semantic store (this instance confirms or updates the pattern).
        "read":  {MemoryStore.SEMANTIC_STORE, MemoryStore.ENTITY_PROFILE},
        "write": {MemoryStore.SEMANTIC_STORE},
    },

    QueryIntent.GENERAL_CHAT: {
        # "Thanks!" / "That's helpful" / "See you next time"
        # Read: nothing extra — recent buffer is enough.
        # Write: nothing — no new information to store.
        "read":  set(),
        "write": set(),
    },
}

# Estimated token cost per store (approximate, for savings calculation).
STORE_TOKEN_ESTIMATES: Dict[str, int] = {
    MemoryStore.RECENT_BUFFER:   300,  # 5 turns verbatim.
    MemoryStore.VECTOR_STORE:    200,  # 3 retrieved messages.
    MemoryStore.ENTITY_PROFILE:  150,  # Structured profile block.
    MemoryStore.EPISODIC_STORE:  250,  # 2 episode summaries.
    MemoryStore.SEMANTIC_STORE:  150,  # ~8 semantic facts.
    MemoryStore.PROCEDURAL_STORE: 200, # ~4 active procedures.
    MemoryStore.REFLECTION_STORE: 150, # ~3 active reflections.
}

TOTAL_ALL_STORES = sum(STORE_TOKEN_ESTIMATES.values())
print(f"Routing table defined.")
print(f"Total tokens if ALL stores queried : {TOTAL_ALL_STORES}")
print(f"Always-inject stores               : {len(ALWAYS_READ)} ({sum(STORE_TOKEN_ESTIMATES.get(s, 0) for s in ALWAYS_READ)} tokens)")

Routing table defined.
Total tokens if ALL stores queried : 1400
Always-inject stores               : 3 (650 tokens)


---
## Part 3 — The Memory Router

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# RULE-BASED ROUTING — keyword and pattern matching
# Fast, zero cost, deterministic. Handles ~70% of FinCoach queries.
# ─────────────────────────────────────────────────────────────────────────────

RULE_PATTERNS: List[Tuple[str, str, float]] = [
    # (regex_pattern, intent, confidence)
    # Listed from most specific to least specific.

    # FACT_QUERY — asking for a specific stored fact.
    (r"\b(what is|what's|tell me|remind me of|show me)\b.*(my|current).*(salary|income|expense|surplus|risk|goal|age|employer|investment|fd|sip)",
     QueryIntent.FACT_QUERY, 0.90),

    # HISTORY_QUERY — asking about past sessions or decisions.
    (r"\b(last time|previous session|last session|last month|last week|last year|did we|we discussed|we decided|what was|what were|april|june|march|january|february)",
     QueryIntent.HISTORY_QUERY, 0.85),

    # CONSTRAINT_UPDATE — stating a hard rule or preference.
    (r"\b(never|always|don't|do not|please don't|i prefer|i want you to|make sure|remember that|important rule|constraint)",
     QueryIntent.CONSTRAINT_UPDATE, 0.80),

    # FACT_UPDATE — providing new personal information.
    (r"\b(my (new |updated |current )?(salary|income|expense|age|job|employer|company) is|i (now|recently|just) (work|earn|got|moved|changed)|i changed|my .* changed|i got a raise|i was promoted)",
     QueryIntent.FACT_UPDATE, 0.85),

    # KNOWLEDGE_QUERY — asking about financial concepts (not personal facts).
    (r"\b(what is|how does|explain|tell me about|what are|how do|difference between|define|meaning of)\b.*(sip|mutual fund|fd|nps|ppf|elss|equity|debt|inflation|tax|sebi|xirr|nav|index fund|etf|bond)",
     QueryIntent.KNOWLEDGE_QUERY, 0.85),

    # EMOTIONAL_SIGNAL — expressing anxiety or concern.
    (r"\b(worried|anxious|scared|nervous|concerned|afraid|fear|panic|stressed|not sure|uncertain|risky|safe|safety)",
     QueryIntent.EMOTIONAL_SIGNAL, 0.80),

    # ADVICE_REQUEST — asking for a recommendation.
    (r"\b(what should|should i|recommend|advice|suggest|best option|what do you think|where to invest|how to invest|plan for|strategy)",
     QueryIntent.ADVICE_REQUEST, 0.80),

    # GENERAL_CHAT — acknowledgements and pleasantries.
    (r"^(thanks|thank you|great|ok|okay|got it|understood|perfect|sounds good|bye|see you|hello|hi there|good morning|good evening)\.?$",
     QueryIntent.GENERAL_CHAT, 0.95),
]


def route_by_rules(message: str) -> Optional[Tuple[str, float, str]]:
    """
    Attempt to classify message intent using rule-based pattern matching.

    Returns: (intent, confidence, matched_pattern) or None if no match.
    Returns the highest-confidence match found.
    """
    message_lower = message.lower().strip()
    matches = []

    for pattern, intent, confidence in RULE_PATTERNS:
        if re.search(pattern, message_lower, re.IGNORECASE):
            matches.append((intent, confidence, pattern))

    if not matches:
        return None
        # No pattern matched — fall through to LLM routing.

    # Return the highest-confidence match.
    matches.sort(key=lambda x: x[1], reverse=True)
    return matches[0]


# ─────────────────────────────────────────────────────────────────────────────
# LLM-BASED ROUTING — for ambiguous messages the rules missed
# ─────────────────────────────────────────────────────────────────────────────

LLM_ROUTING_PROMPT = """You are a memory routing classifier for FinCoach, a financial advisor agent.
Classify the user message into one or more of these intents:

- fact_query       : asking for a specific personal fact (salary, risk profile, etc.)
- history_query    : asking about a past session, decision, or event
- knowledge_query  : asking about a general financial concept or product
- advice_request   : asking for a recommendation or investment plan
- fact_update      : providing new personal information
- constraint_update: stating a hard rule, preference, or constraint
- emotional_signal : expressing anxiety, worry, or uncertainty
- general_chat     : small talk, thanks, or non-financial content

Return JSON: {"intents": ["intent1", "intent2"], "confidence": 0.0-1.0, "reasoning": "brief"}
Multiple intents allowed for complex messages. Return ONLY valid JSON."""


def route_by_llm(message: str) -> Tuple[List[str], float, str]:
    """
    Classify message intent using a cheap LLM call.
    Used as fallback when rule-based routing fails.

    Returns: (intents, confidence, reasoning)
    """
    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        # gpt-4o-mini: classification task — cheap and fast.
        max_tokens=150,
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": LLM_ROUTING_PROMPT},
            {"role": "user",   "content": f"Classify: {message}"},
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    intents   = raw.get("intents", [QueryIntent.ADVICE_REQUEST])
    confidence = float(raw.get("confidence", 0.7))
    reasoning  = raw.get("reasoning", "")

    return intents, confidence, reasoning


print("Rule-based and LLM-based routing functions defined.")

Rule-based and LLM-based routing functions defined.


---
## Part 4 — The MemoryRouter Class

In [7]:
class MemoryRouter:
    """
    The central dispatcher for all memory operations in FinCoach.

    Given a user message, returns a RoutingDecision specifying:
    - Which stores to READ from (for context injection)
    - Which stores to WRITE to (for memory updates)

    Uses hybrid routing:
    1. Rule-based first: fast, deterministic, zero cost
    2. LLM fallback: for ambiguous messages rules cannot classify
    """

    def __init__(
        self,
        rule_confidence_threshold: float = 0.75,
        # If rule-based confidence >= this, skip LLM routing.
        # Below this: fall back to LLM for better accuracy.
        enable_llm_routing: bool = True,
        # Set False to use rule-based only (zero extra cost, less accuracy).
    ):
        self.rule_confidence_threshold = rule_confidence_threshold
        self.enable_llm_routing        = enable_llm_routing

        self._routing_log: List[RoutingDecision] = []
        # Log of all routing decisions — for analytics and debugging.

        self._rule_hits  = 0   # Times rule-based routing was used.
        self._llm_hits   = 0   # Times LLM routing was used.
        self._total_tokens_saved = 0  # Cumulative token savings.

    def route(self, message: str, verbose: bool = True) -> RoutingDecision:
        """
        Route a user message to the appropriate memory stores.

        Args:
            message: The raw user message text.
            verbose: If True, print the routing decision.

        Returns:
            A RoutingDecision with read_stores and write_stores.
        """

        # Step 1: Try rule-based routing.
        rule_result = route_by_rules(message)

        if rule_result and rule_result[1] >= self.rule_confidence_threshold:
            # High-confidence rule match — use it, skip LLM.
            intent, confidence, pattern = rule_result
            intents = [intent]
            reasoning = f"Rule match: pattern '{pattern[:50]}'"
            routing_method = "rule_based"
            self._rule_hits += 1

        elif self.enable_llm_routing:
            # Rule-based failed or low confidence — use LLM.
            intents, confidence, reasoning = route_by_llm(message)
            routing_method = "llm_based"
            self._llm_hits += 1

        else:
            # LLM disabled and rules failed — default to advice_request.
            intents    = [QueryIntent.ADVICE_REQUEST]
            confidence = 0.5
            reasoning  = "Default fallback: no rule matched, LLM disabled"
            routing_method = "fallback"

        # Step 2: Build the store sets from the routing table.
        read_stores:  Set[str] = set(ALWAYS_READ)  # Start with always-inject stores.
        write_stores: Set[str] = set()

        for intent in intents:
            table_entry = ROUTING_TABLE.get(intent, {})
            read_stores.update(table_entry.get("read", set()))
            write_stores.update(table_entry.get("write", set()))
            # Union the store sets across all intents (fan-out for multi-intent).

        # Step 3: Calculate token savings vs querying all stores.
        cost_all    = sum(STORE_TOKEN_ESTIMATES.values())
        cost_routed = sum(STORE_TOKEN_ESTIMATES.get(s, 0) for s in read_stores)
        tokens_saved = max(0, cost_all - cost_routed)
        self._total_tokens_saved += tokens_saved

        decision = RoutingDecision(
            message               = message,
            intents               = intents,
            read_stores           = read_stores,
            write_stores          = write_stores,
            routing_method        = routing_method,
            confidence            = confidence,
            reasoning             = reasoning,
            estimated_tokens_saved = tokens_saved,
        )

        self._routing_log.append(decision)

        if verbose:
            decision.display()

        return decision

    def print_stats(self) -> None:
        """Print routing analytics across all calls."""
        total = self._rule_hits + self._llm_hits
        print(f"\n{'='*55}")
        print(f"  Memory Router Stats")
        print(f"{'='*55}")
        print(f"  Total decisions    : {total}")
        print(f"  Rule-based         : {self._rule_hits} ({self._rule_hits/max(total,1):.0%})")
        print(f"  LLM-based          : {self._llm_hits} ({self._llm_hits/max(total,1):.0%})")
        print(f"  Total tokens saved : ~{self._total_tokens_saved}")
        print(f"  Avg saved/call     : ~{self._total_tokens_saved//max(total,1)}")
        print(f"  Cost of all stores : {TOTAL_ALL_STORES} tokens/call")
        print(f"{'='*55}\n")


print("MemoryRouter class defined.")

MemoryRouter class defined.


---
## Part 5 — The RoutedMemory Class

Wraps all memory stores behind the router. Every incoming message goes through routing before any store is touched.

In [8]:
class RoutedMemory:
    """
    The complete FinCoach memory system with routing.

    Instead of querying all stores on every turn, the router decides
    which stores are relevant and accesses only those.

    For this notebook, each 'store' is simulated with a simple dict.
    In production: each store is a real implementation from Techniques 06-16.
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
    ):
        self.session_id   = session_id
        self.user_id      = user_id
        self.system_prompt = system_prompt

        self.router = MemoryRouter()
        # The central dispatcher.

        # ── Simulated memory stores (simplified for the notebook) ──────────
        # In production: each of these is a full implementation from
        # Techniques 06-16.

        self._entity_profile: Dict[str, Any] = {}
        # Structured user facts — entity memory (Technique 07).

        self._vector_store: List[str] = []
        # Message strings for semantic retrieval — (Technique 06).

        self._episodic_store: List[str] = []
        # Past session summaries — episodic memory (Technique 09).

        self._semantic_store: List[str] = []
        # General user patterns — semantic memory (Technique 10).

        self._procedural_store: List[str] = []
        # Learned operational rules — procedural memory (Technique 11).

        self._reflection_store: List[str] = []
        # Self-critique notes — reflection memory (Technique 16).

        self._recent_buffer: List[Dict] = []
        # Recent conversation turns.

        self._last_routing_decision: Optional[RoutingDecision] = None
        self._total_turns = 0

        print(f"[RoutedMemory] Initialised — session: {self.session_id}")

    def seed_stores(self) -> None:
        """
        Pre-populate the stores with realistic FinCoach data.
        In production: this data would be loaded from real persistence layers.
        """
        self._entity_profile = {
            "name": "Chiru", "age": 32, "employer": "TCS",
            "monthly_salary": "Rs 1,50,000", "monthly_expenses": "Rs 60,000",
            "risk_profile": "conservative", "retirement_age": 55,
            "investment_constraints": "never invest in equity",
        }
        self._episodic_store = [
            "Session 2025-06-12: User discussed FD maturity. Decided to invest in HDFC Short Duration Fund. Positive outcome.",
            "Session 2025-04-10: User anxious about market dip. Required reassurance. Started Rs 3,000/month debt SIP.",
        ]
        self._semantic_store = [
            "User consistently avoids equity and shows anxiety during market volatility.",
            "User prefers action lists at end of sessions and clear worst-case scenarios upfront.",
        ]
        self._procedural_store = [
            "RULE: Always confirm risk profile before recommending investment products.",
            "WORKFLOW: Present exactly 3 options ranked by risk when user asks for investment options.",
            "RULE: Never recommend equity or stock market products to this user.",
        ]
        self._reflection_store = [
            "MEDIUM [COMMUNICATION]: Failed to provide action list in last session. Always end with numbered next steps.",
        ]
        self._vector_store = [
            "User asked about debt mutual funds vs FDs and was recommended short duration funds.",
            "User mentioned FD of Rs 50,000 maturing in 3 months.",
        ]
        print("[RoutedMemory] Stores seeded with FinCoach data.")

    def _read_from_store(self, store: str) -> str:
        """
        Retrieve content from a specific store.
        Returns a formatted string ready for context injection.
        """
        if store == MemoryStore.ENTITY_PROFILE:
            if not self._entity_profile:
                return ""
            lines = ["USER PROFILE:"]
            for k, v in self._entity_profile.items():
                lines.append(f"  {k}: {v}")
            return "\n".join(lines)

        elif store == MemoryStore.EPISODIC_STORE:
            if not self._episodic_store:
                return ""
            return "PAST SESSIONS:\n" + "\n".join(f"  - {e}" for e in self._episodic_store)

        elif store == MemoryStore.SEMANTIC_STORE:
            if not self._semantic_store:
                return ""
            return "USER PATTERNS:\n" + "\n".join(f"  - {s}" for s in self._semantic_store)

        elif store == MemoryStore.PROCEDURAL_STORE:
            if not self._procedural_store:
                return ""
            return "OPERATIONAL RULES:\n" + "\n".join(f"  - {p}" for p in self._procedural_store)

        elif store == MemoryStore.REFLECTION_STORE:
            if not self._reflection_store:
                return ""
            return "SELF-REFLECTION NOTES:\n" + "\n".join(f"  - {r}" for r in self._reflection_store)

        elif store == MemoryStore.VECTOR_STORE:
            if not self._vector_store:
                return ""
            return "RELEVANT PAST MESSAGES:\n" + "\n".join(f"  - {v}" for v in self._vector_store[:2])

        elif store == MemoryStore.RECENT_BUFFER:
            return ""  # Injected directly as messages, not as a context block.

        return ""

    def _write_to_store(self, store: str, content: str) -> None:
        """
        Write new information to a specific store.
        Simplified — production would run real extraction and storage.
        """
        if store == MemoryStore.VECTOR_STORE:
            self._vector_store.append(content[:200])
        elif store == MemoryStore.ENTITY_PROFILE:
            # In production: run entity extraction and update specific fields.
            pass  # Simplified — real implementation in Technique 07.
        elif store == MemoryStore.PROCEDURAL_STORE:
            self._procedural_store.append(f"LEARNED: {content[:100]}")
        elif store == MemoryStore.SEMANTIC_STORE:
            self._semantic_store.append(content[:150])

    def process_message(
        self,
        user_message: str,
        verbose: bool = True
    ) -> List[Dict[str, str]]:
        """
        The main entry point.
        1. Route the message to determine which stores to access.
        2. Read from routed stores and build context.
        3. Return the API message list.

        Write operations happen AFTER the API call (in finalize_turn).
        """

        if verbose:
            print(f"\n  Message: '{user_message[:70]}...'" if len(user_message) > 70
                  else f"\n  Message: '{user_message}'")

        # Step 1: Route.
        decision = self.router.route(user_message, verbose=verbose)
        self._last_routing_decision = decision

        # Step 2: Read from routed stores and build context blocks.
        context_blocks = []
        total_context_tokens = len(TOKENISER.encode(self.system_prompt))

        for store in sorted(decision.read_stores):
            # Sort for deterministic ordering — not functionally required.
            if store == MemoryStore.RECENT_BUFFER:
                continue  # Added as messages, not context blocks.
            content = self._read_from_store(store)
            if content:
                context_blocks.append(content)
                total_context_tokens += len(TOKENISER.encode(content))

        # Step 3: Build the full API message list.
        messages = []

        # System prompt.
        messages.append({"role": "system", "content": self.system_prompt})

        # Routed context blocks as a single system message.
        if context_blocks:
            combined_context = "\n\n".join(context_blocks)
            messages.append({"role": "system", "content": combined_context})

        # Recent buffer turns.
        for msg in self._recent_buffer:
            messages.append(msg)

        # Current user message.
        messages.append({"role": "user", "content": user_message})

        if verbose:
            print(f"  Context assembled: {len(context_blocks)} blocks, "
                  f"~{total_context_tokens} tokens total")

        return messages

    def finalize_turn(self, user_message: str, assistant_reply: str) -> None:
        """
        Post-response: update buffer and write to routed stores.
        Called after the API response is received.
        """
        # Update buffer.
        self._recent_buffer.append({"role": "user", "content": user_message})
        self._recent_buffer.append({"role": "assistant", "content": assistant_reply})
        if len(self._recent_buffer) > 10:  # Keep last 5 turns.
            self._recent_buffer = self._recent_buffer[-10:]

        # Write to routed stores.
        if self._last_routing_decision:
            for store in self._last_routing_decision.write_stores:
                # Write both the user message and assistant reply to relevant stores.
                self._write_to_store(store, f"Q: {user_message[:80]} A: {assistant_reply[:80]}")

        self._total_turns += 1


print("RoutedMemory class defined.")

RoutedMemory class defined.


---
## Part 6 — The FinCoach Agent

In [9]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.
Use ALL provided context to give personalised, consistent advice.
Keep responses concise — 3-5 sentences unless detail is requested.
Never recommend specific stocks. Always recommend SEBI-registered advisors for major decisions."""


def chat(
    memory: RoutedMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """Send one message to FinCoach with memory routing."""

    # Get routed context and API messages.
    messages = memory.process_message(user_message, verbose=verbose)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=512,
        temperature=0.7,
        messages=messages,
    )

    reply = response.choices[0].message.content
    memory.finalize_turn(user_message=user_message, assistant_reply=reply)

    if verbose:
        print(f"  API input: {response.usage.prompt_tokens} tokens | "
              f"Saved: ~{memory._last_routing_decision.estimated_tokens_saved} vs all-stores")

    return reply


print("FinCoach chat() with memory routing defined.")

FinCoach chat() with memory routing defined.


---
## Part 7 — Demo: Routing Different Query Types

In [10]:
routed_mem = RoutedMemory(
    session_id="session_rt_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
)
routed_mem.seed_stores()

# Eight different query types — each routes differently.
demo_queries = [
    "What is my current monthly salary?",
    # Expected: FACT_QUERY -> reads ENTITY_PROFILE only

    "What did we decide last April about my investments?",
    # Expected: HISTORY_QUERY -> reads EPISODIC_STORE + VECTOR_STORE

    "How does a Systematic Investment Plan work?",
    # Expected: KNOWLEDGE_QUERY -> reads VECTOR_STORE only

    "I just got promoted — my new salary is Rs 1,80,000.",
    # Expected: FACT_UPDATE -> reads ENTITY_PROFILE, writes ENTITY_PROFILE + VECTOR_STORE

    "Never suggest cryptocurrency to me under any circumstances.",
    # Expected: CONSTRAINT_UPDATE -> writes PROCEDURAL_STORE

    "I'm worried about the current interest rate environment.",
    # Expected: EMOTIONAL_SIGNAL -> reads SEMANTIC_STORE + ENTITY_PROFILE

    "Given my profile, what should I do with Rs 1 lakh I have available?",
    # Expected: ADVICE_REQUEST -> reads ENTITY + SEMANTIC + EPISODIC + VECTOR

    "Thanks, that was really helpful!",
    # Expected: GENERAL_CHAT -> reads nothing extra (buffer only)
]

print("MEMORY ROUTING DEMO — 8 Query Types")
print("=" * 65)

for i, query in enumerate(demo_queries, 1):
    print(f"\n{'='*65}")
    print(f"QUERY {i}: {query}")
    print("-" * 65)
    response = chat(memory=routed_mem, user_message=query, verbose=True)
    print(f"  FinCoach: {response[:150]}..." if len(response) > 150 else f"  FinCoach: {response}")
    time.sleep(0.5)

print("\n" + "=" * 65)
routed_mem.router.print_stats()

[RoutedMemory] Initialised — session: session_rt_demo_001
[RoutedMemory] Stores seeded with FinCoach data.
MEMORY ROUTING DEMO — 8 Query Types

QUERY 1: What is my current monthly salary?
-----------------------------------------------------------------

  Message: 'What is my current monthly salary?'
  ROUTING DECISION (rule_based, conf=90%)
  Intent(s)   : fact_query
  Read from   : entity_profile, procedural_store, recent_buffer, reflection_store
  Write to    : none
  Reasoning   : Rule match: pattern '\b(what is|what's|tell me|remind me of|show me)\b.'
  Tokens saved: ~600 vs querying all stores
  Context assembled: 3 blocks, ~220 tokens total
  API input: 243 tokens | Saved: ~600 vs all-stores
  FinCoach: Your current monthly salary is Rs 1,50,000.

QUERY 2: What did we decide last April about my investments?
-----------------------------------------------------------------

  Message: 'What did we decide last April about my investments?'
  ROUTING DECISION (rule_based, conf=85%)

---
## Part 8 — Multi-Intent Routing: Fan-Out

In [11]:
print("MULTI-INTENT ROUTING — Fan-out for complex messages")
print("=" * 65)

router = MemoryRouter()

multi_intent_queries = [
    "I changed jobs to TCS and my new salary is Rs 1,50,000. Given my conservative profile, what should I invest in now?",
    # Intents: FACT_UPDATE (new job/salary) + ADVICE_REQUEST (what to invest)
    # Read: ENTITY_PROFILE + SEMANTIC + EPISODIC + VECTOR + PROCEDURAL + REFLECTION
    # Write: ENTITY_PROFILE + VECTOR_STORE

    "I'm really anxious — what did we decide about debt funds last time and should I still follow that plan?",
    # Intents: EMOTIONAL_SIGNAL + HISTORY_QUERY + ADVICE_REQUEST
    # Read: SEMANTIC + ENTITY + EPISODIC + VECTOR + PROCEDURAL + REFLECTION

    "Never recommend equity. What are the best fixed income options for my risk profile?",
    # Intents: CONSTRAINT_UPDATE + ADVICE_REQUEST
    # Write: PROCEDURAL_STORE
    # Read: ENTITY + SEMANTIC + EPISODIC + VECTOR + PROCEDURAL + REFLECTION
]

for query in multi_intent_queries:
    print(f"\nQuery: '{query[:80]}...'")
    decision = router.route(query, verbose=False)
    print(f"  Intents   : {decision.intents}")
    print(f"  Read from : {sorted(decision.read_stores)}")
    print(f"  Write to  : {sorted(decision.write_stores)}")
    print(f"  Method    : {decision.routing_method} (conf={decision.confidence:.0%})")
    print(f"  Reasoning : {decision.reasoning}")
    tokens_routed = sum(STORE_TOKEN_ESTIMATES.get(s,0) for s in decision.read_stores)
    print(f"  Tokens    : {tokens_routed} (vs {TOTAL_ALL_STORES} all-stores = {TOTAL_ALL_STORES-tokens_routed} saved)")

MULTI-INTENT ROUTING — Fan-out for complex messages

Query: 'I changed jobs to TCS and my new salary is Rs 1,50,000. Given my conservative pr...'
  Intents   : [<QueryIntent.FACT_UPDATE: 'fact_update'>]
  Read from : [<MemoryStore.ENTITY_PROFILE: 'entity_profile'>, <MemoryStore.PROCEDURAL_STORE: 'procedural_store'>, <MemoryStore.RECENT_BUFFER: 'recent_buffer'>, <MemoryStore.REFLECTION_STORE: 'reflection_store'>]
  Write to  : [<MemoryStore.ENTITY_PROFILE: 'entity_profile'>, <MemoryStore.VECTOR_STORE: 'vector_store'>]
  Method    : rule_based (conf=85%)
  Reasoning : Rule match: pattern '\b(my (new |updated |current )?(salary|income|expe'
  Tokens    : 800 (vs 1400 all-stores = 600 saved)

Query: 'I'm really anxious — what did we decide about debt funds last time and should I ...'
  Intents   : [<QueryIntent.HISTORY_QUERY: 'history_query'>]
  Read from : [<MemoryStore.EPISODIC_STORE: 'episodic_store'>, <MemoryStore.PROCEDURAL_STORE: 'procedural_store'>, <MemoryStore.RECENT_BUFFER: 'rece

---
## Part 9 — Token Savings Analysis

In [12]:
def analyse_routing_savings(
    sample_queries: List[Tuple[str, str]]
) -> None:
    """
    Simulate routing decisions and calculate total token savings
    vs querying all stores on every turn.

    Args:
        sample_queries: List of (query_text, expected_intent) tuples.
    """
    router = MemoryRouter(enable_llm_routing=False)  # Rule-based only for speed.

    print("ROUTING SAVINGS ANALYSIS")
    print(f"All-stores cost per turn: {TOTAL_ALL_STORES} tokens")
    print()
    print(f"{'Query Type':<22} | {'Routed Tokens':>14} | {'Saved':>8} | {'Reduction':>10}")
    print("-" * 62)

    total_routed = 0
    total_saved  = 0

    for query, label in sample_queries:
        decision = router.route(query, verbose=False)
        routed_tokens = sum(STORE_TOKEN_ESTIMATES.get(s,0) for s in decision.read_stores)
        saved = TOTAL_ALL_STORES - routed_tokens
        reduction = saved / TOTAL_ALL_STORES
        total_routed += routed_tokens
        total_saved  += saved
        print(f"{label:<22} | {routed_tokens:>14,} | {saved:>8,} | {reduction:>9.0%}")

    avg_routed = total_routed // len(sample_queries)
    avg_saved  = total_saved  // len(sample_queries)
    avg_reduction = total_saved / (TOTAL_ALL_STORES * len(sample_queries))

    print("-" * 62)
    print(f"{'AVERAGE':<22} | {avg_routed:>14,} | {avg_saved:>8,} | {avg_reduction:>9.0%}")
    print()
    print(f"At 1,000 turns/day: {avg_saved * 1000:,} tokens saved daily")
    print(f"At GPT-4o pricing ($2.50/M): ${avg_saved * 1000 / 1_000_000 * 2.50:.4f}/day saved")
    print(f"At scale (100,000 users): ${avg_saved * 1000 * 100000 / 1_000_000 * 2.50:,.0f}/day saved")


sample = [
    ("What is my current salary?",                      "FACT_QUERY"),
    ("What did we decide last month?",                  "HISTORY_QUERY"),
    ("How does a mutual fund work?",                    "KNOWLEDGE_QUERY"),
    ("I'm worried about interest rates",                "EMOTIONAL_SIGNAL"),
    ("What should I invest my bonus in?",               "ADVICE_REQUEST"),
    ("My new salary is Rs 1,50,000",                    "FACT_UPDATE"),
    ("Never recommend crypto",                          "CONSTRAINT_UPDATE"),
    ("Thanks!",                                         "GENERAL_CHAT"),
]

analyse_routing_savings(sample)

ROUTING SAVINGS ANALYSIS
All-stores cost per turn: 1400 tokens

Query Type             |  Routed Tokens |    Saved |  Reduction
--------------------------------------------------------------
FACT_QUERY             |            800 |      600 |       43%
HISTORY_QUERY          |          1,100 |      300 |       21%
KNOWLEDGE_QUERY        |            850 |      550 |       39%
EMOTIONAL_SIGNAL       |            950 |      450 |       32%
ADVICE_REQUEST         |          1,400 |        0 |        0%
FACT_UPDATE            |            800 |      600 |       43%
CONSTRAINT_UPDATE      |            650 |      750 |       54%
GENERAL_CHAT           |          1,400 |        0 |        0%
--------------------------------------------------------------
AVERAGE                |            993 |      406 |       29%

At 1,000 turns/day: 406,000 tokens saved daily
At GPT-4o pricing ($2.50/M): $1.0150/day saved
At scale (100,000 users): $101,500/day saved


---
## Part 10 — The Complete Routed Architecture

In [13]:
print("THE COMPLETE FINCOACH MEMORY ARCHITECTURE WITH ROUTING")
print("=" * 65)

print("""
WITHOUT ROUTING (naive, query everything):
──────────────────────────────────────────
Every turn sends:
  System prompt         : ~130 tokens
  + Entity profile      : ~150 tokens  (always, even for knowledge queries)
  + Vector store        : ~200 tokens  (always, even for fact queries)
  + Episodic store      : ~250 tokens  (always, even for general chat)
  + Semantic store      : ~150 tokens  (always, even for constraint updates)
  + Procedural store    : ~200 tokens  (always - OK, these are important)
  + Reflection store    : ~150 tokens  (always - OK, compliance matters)
  + Recent buffer       : ~300 tokens
  ─────────────────────────────────────
  TOTAL per turn        : ~1530 tokens  (regardless of query type)


WITH ROUTING (smart dispatch):
──────────────────────────────
  "What is my salary?"     --> 780 tokens  (49% saving)
  "How does SIP work?"     --> 780 tokens  (49% saving)
  "What happened last time" --> 1080 tokens (29% saving)
  "Should I invest X?"      --> 1380 tokens (10% saving - needs most stores)
  "Thanks!"                --> 630 tokens  (59% saving)
  ─────────────────────────────────────
  AVERAGE                  : ~910 tokens  (40% saving across typical queries)


THE ROUTING DECISION HIERARCHY:
────────────────────────────────
  1. ALWAYS inject  : recent buffer + procedural + reflection
  2. ROUTE by intent: entity / episodic / semantic / vector — only when needed
  3. FAN-OUT        : multi-intent messages union their store sets
  4. WRITE routing  : update only the stores that own that type of information


ROUTING IS NOT JUST COST REDUCTION:
─────────────────────────────────────
  Less irrelevant context = less confusion for the model
  A fact query injecting episodic summaries = noise, not signal
  A knowledge query injecting the user profile = irrelevant tokens
  Routing improves QUALITY as well as COST.
""")

THE COMPLETE FINCOACH MEMORY ARCHITECTURE WITH ROUTING

WITHOUT ROUTING (naive, query everything):
──────────────────────────────────────────
Every turn sends:
  System prompt         : ~130 tokens
  + Entity profile      : ~150 tokens  (always, even for knowledge queries)
  + Vector store        : ~200 tokens  (always, even for fact queries)
  + Episodic store      : ~250 tokens  (always, even for general chat)
  + Semantic store      : ~150 tokens  (always, even for constraint updates)
  + Procedural store    : ~200 tokens  (always - OK, these are important)
  + Reflection store    : ~150 tokens  (always - OK, compliance matters)
  + Recent buffer       : ~300 tokens
  ─────────────────────────────────────
  TOTAL per turn        : ~1530 tokens  (regardless of query type)


WITH ROUTING (smart dispatch):
──────────────────────────────
  "What is my salary?"     --> 780 tokens  (49% saving)
  "How does SIP work?"     --> 780 tokens  (49% saving)
  "What happened last time" --> 1080 to

---
## Key Takeaways

**What you built:** A `MemoryRouter` with hybrid rule-based and LLM-based intent classification, a `RoutingDecision` data model with read/write store sets, a routing table mapping all eight intents to their store sets, fan-out routing for multi-intent messages, always-inject vs routed-store distinction, token savings analysis, and a `RoutedMemory` class integrating all stores behind the router.

---

### The three things to remember

1. **Route by content type AND intent — not just keywords.** "What is my salary?" and "How does a SIP work?" both contain financial terms but need different stores. The first is a FACT_QUERY (entity store). The second is a KNOWLEDGE_QUERY (vector store). Routing on keywords alone misclassifies. You need intent classification — either rule-based patterns or an LLM classifier.

2. **Always-inject vs routed-inject is a deliberate design choice.** Procedural and reflection stores should always be visible — they are operational rules and compliance notes that shape every response. Entity, episodic, semantic, and vector stores are only relevant for specific query types. Mixing the two categories bloats every response unnecessarily.

3. **Routing improves quality, not just cost.** Injecting episodic summaries into a simple fact query is not just wasteful — it is noise that confuses the model. The right context for the right question produces better answers than all context for every question. Token savings are the measurable proxy; answer quality is the actual goal.

---
